<a href="https://colab.research.google.com/github/Ravinduranasinghe/Adv-Auto-Filter-Bot-V2/blob/main/yt_dlp_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, subprocess, re
from google.colab import drive, files

# ============================================================
#  CELL 1 — Setup
# ============================================================
def setup():
    print("🔧 Installing Node.js 20...")
    subprocess.run('curl -fsSL https://deb.nodesource.com/setup_20.x | bash -', shell=True, capture_output=True)
    subprocess.run(['apt-get', 'install', '-y', 'nodejs'], capture_output=True)

    print("🔧 Updating yt-dlp...")
    subprocess.run(['pip', 'install', '-U', '--pre', 'yt-dlp', '--break-system-packages'], check=True)

    yt   = subprocess.run(['yt-dlp', '--version'], capture_output=True, text=True)
    node = subprocess.run(['node',   '--version'], capture_output=True, text=True)
    print(f"✅ yt-dlp {yt.stdout.strip()} | node {node.stdout.strip()}")
    print("✅ Setup complete.")

setup()

# ============================================================
#  CONFIGURATION
# ============================================================
playlist_url = "https://www.youtube.com/playlist?list=PLKr-lYdiK6lLtGvzhCEy0APOzqLqK2viE"

# ============================================================
#  STEP 1 — Upload cookies.txt
# ============================================================
print("\n📂 Upload your 'cookies.txt' file.")
uploaded = files.upload()
if not uploaded:
    raise Exception("No cookie file uploaded.")
cookie_file = list(uploaded.keys())[0]
print(f"✅ Cookie file: {cookie_file}")

# ============================================================
#  STEP 2 — Mount Google Drive
# ============================================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("✅ Drive already mounted.")

# ============================================================
#  STEP 3 — Fetch playlist title & create save folder
# ============================================================
print("\n🔍 Fetching playlist title...")
get_title = subprocess.run(
    ['yt-dlp',
     '--cookies', cookie_file,
     '--extractor-args', 'youtube:player_client=web;player_skip=webpage',
     '--js-runtimes', 'node',
     '--get-filename', '-o', '%(playlist_title)s',
     '--playlist-items', '1',
     playlist_url],
    capture_output=True, text=True
)
raw_title  = get_title.stdout.strip() or "Playlist"
clean_name = re.sub(r'[\\/*?:"<>|]', "_", raw_title)
save_path  = f"/content/drive/MyDrive/{clean_name}"
os.makedirs(save_path, exist_ok=True)
print(f"✅ Saving to: Drive/{clean_name}")

# ============================================================
#  STEP 4 — Download + transcode to 48k opus
# ============================================================
print(f"\n⬇️  Downloading… (existing files will be skipped)\n")

cmd = [
    'yt-dlp',
    '--cookies',            cookie_file,

    # web client supports cookies; player_skip=webpage avoids the SABR
    # experiment that strips format URLs from web_creator responses
    '--extractor-args',    'youtube:player_client=web;player_skip=webpage',
    '--js-runtimes',       'node',

    # ── Best audio, transcode to 48k opus ───────────────────
    '-f',                  'bestaudio/best',
    '-x',
    '--audio-format',      'opus',
    '--audio-quality',     '10',
    '--postprocessor-args', (
        'FFmpegExtractAudio:'
        '-c:a libopus '
        '-b:a 48k '
        '-vbr on '
        '-compression_level 10 '
        '-application audio'
    ),

    # ── Metadata ─────────────────────────────────────────────
    '--add-metadata',
    '--no-embed-thumbnail',

    # ── File handling ─────────────────────────────────────────
    '--windows-filenames',
    '--no-overwrites',
    '--no-abort-on-error',

    # ── Rate limiting ─────────────────────────────────────────
    '--sleep-interval',     '5',
    '--max-sleep-interval', '12',

    # ── Output ───────────────────────────────────────────────
    '-o', f'{save_path}/%(playlist_index)s - %(title)s.%(ext)s',

    playlist_url
]

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

# ============================================================
#  CLEANUP
# ============================================================
if os.path.exists(cookie_file):
    os.remove(cookie_file)
    print("\n🗑️  Cookie file deleted.")

print(f"\n✅ Done! Files saved to: Drive/{clean_name}")